# บทที่ 1 · รู้จักกับ pandas

**pandas** คือไลบรารีของ Python สำหรับ "จับตารางมาเล่น" — อ่านข้อมูล คัดกรอง คำนวณ และสรุปผล ได้เร็วกว่าการเขียนวนลูปเองหลายสิบเท่า ชื่อของมันไม่เกี่ยวกับหมีแพนด้า แต่มาจากคำว่า *panel data* (ข้อมูลแบบตาราง) ในสายสถิติ

| โครงสร้าง | มิติ | เทียบให้เห็นภาพ |
|---|---|---|
| `Series` | 1 มิติ | คอลัมน์เดียวของตาราง เหมือนรายการราคาในใบเมนู |
| `DataFrame` | 2 มิติ | ตารางเต็มรูปแบบ (แถว × คอลัมน์) เหมือนชีต Excel |

## 🗂️ การ์ดข้อมูล: เราจะใช้ข้อมูลอะไรบ้าง (อ่านก่อนใช้เสมอ!)

ในบทนี้เราเริ่มใช้ **ข้อมูลสภาพอากาศจริง** ไฟล์ `data/thai_monthly_climate_1991_2020.csv`

| รายการ | รายละเอียด |
|---|---|
| **ที่มา** | ค่าปกติ 30 ปี (พ.ศ. 2534–2563 / ค.ศ. 1991–2020) ของกรมอุตุนิยมวิทยา (TMD) เผยแพร่ผ่าน WMO และตารางค่าปกติมาตรฐาน |
| **สถานีที่เลือก** | เชียงใหม่ (CM) · กรุงเทพมหานคร (BKK) · ภูเก็ต (PHKT) — เลือกให้เห็นภาพต่างกันชัด: เหนือ / กลาง / ใต้ |
| **จำนวนแถว** | 12 แถว (เดือน ม.ค.–ธ.ค.) |
| **ความหมายคอลัมน์** | `month` เลขเดือน 1–12 · `month_en`/`month_th` ชื่อเดือน · `*_mean_c` อุณหภูมิเฉลี่ยรายวัน (°C) · `*_rain_mm` ปริมาณฝนรวม (มม.) · `*_rain_days` จำนวนวันที่ฝนตก (วัน) |
| **ตัวอักษรย่อ** | `cm` = เชียงใหม่, `bkk` = กรุงเทพฯ, `phkt` = ภูเก็ต |
| **สิ่งที่ควรรู้ก่อนใช้** | ค่าปกติ 30 ปีคือ "ค่าเฉลี่ยระยะยาว" ไม่ใช่พยากรณ์อากาศปีนี้ · ข้อมูลครบถ้วนไม่มีค่าหาย · หน่วยอุณหภูมิเป็นองศาเซลเซียส ฝนเป็นมิลลิเมตร |
| **สิทธิ์การใช้** | ข้อมูลอุตุนิยมวิทยาสาธารณะ ใช้เพื่อการเรียนการสอนได้ พร้อมอ้างอิงแหล่งที่มา |

> 💡 **นักวิเคราะห์มืออาชีพ** ได้ไฟล์ใหม่ทุกครั้งจะ "แง้มดู" ก่อนเสมอ — บทนี้จะสอนคำสั่งสำรวจพอดี

In [15]:
# อ่านไฟล์ CSV จริงเป็น DataFrame — คำสั่งที่ใช้บ่อยที่สุดในชีวิตนักวิเคราะห์
import pandas as pd

df = pd.read_csv("data/thai_monthly_climate_1991_2020.csv")

print(df.head())        # ดู 5 แถวแรก
print()
print("ขนาดตาราง (แถว, คอลัมน์):", df.shape)

   month month_en month_th  cm_mean_c  bkk_mean_c  phkt_mean_c  cm_rain_mm  \
0      1      Jan     ม.ค.       22.0        27.4         28.3        11.3   
1      2      Feb     ก.พ.       24.2        28.6         29.0         9.6   
2      3      Mar    มี.ค.       27.3        29.7         29.4        20.4   
3      4      Apr    เม.ย.       29.5        30.7         29.6        54.3   
4      5      May     พ.ค.       28.6        30.3         29.1       167.5   

   bkk_rain_mm  phkt_rain_mm  cm_rain_days  bkk_rain_days  phkt_rain_days  
0         23.6          49.9           1.0            1.9             4.3  
1         21.4          24.0           0.7            1.9             2.2  
2         51.0          80.7           2.1            3.4             6.0  
3         93.3         135.1           4.7            5.4             9.6  
4        216.8         236.9          12.0           12.4            15.7  

ขนาดตาราง (แถว, คอลัมน์): (12, 12)


สังเกตคอลัมน์ทั้ง 12: แต่ละเมืองมี 3 คอลัมน์ (อุณหภูมิ / ฝน / วันฝนตก) — โครงสร้างแบบนี้เรียกว่า **wide format** อ่านง่ายสำหรับคน และเหมาะกับการวาดกราฟเทียบกันในบทที่ 4

ต่อไปลองหยิบ "คอลัมน์เดียว" ออกมา — นั่นคือ **Series**

In [16]:
# เลือกคอลัมน์เดียว -> ได้ Series (ข้อมูล 1 มิติ มี index กำกับ)
cm_temp = df["cm_mean_c"]

print(cm_temp)
print()
print("ค่าเฉลี่ยทั้งปีของเชียงใหม่ :", cm_temp.mean(), "°C")
print("เดือนที่ร้อนที่สุด (ค่าสูงสุด) :", cm_temp.max(), "°C")
print("เดือนที่เย็นที่สุด (ค่าต่ำสุด) :", cm_temp.min(), "°C")

0     22.0
1     24.2
2     27.3
3     29.5
4     28.6
5     28.0
6     27.4
7     26.9
8     26.9
9     26.4
10    24.6
11    22.2
Name: cm_mean_c, dtype: float64

ค่าเฉลี่ยทั้งปีของเชียงใหม่ : 26.166666666666668 °C
เดือนที่ร้อนที่สุด (ค่าสูงสุด) : 29.5 °C
เดือนที่เย็นที่สุด (ค่าต่ำสุด) : 22.0 °C


Series มี index 0–11 กำกับ (ตรงกับเดือนที่ 1–12) และมีสถิติสำเร็จรูปให้เรียกใช้ทันที `.mean()` `.max()` `.min()`

สุดท้าย คำสั่งสำรวจมาตรฐานที่ต้องรันทุกครั้ง: `info()` และ `describe()`

In [17]:
# info() : ชนิดข้อมูล + ค่าหายของทุกคอลัมน์
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   month           12 non-null     int64  
 1   month_en        12 non-null     str    
 2   month_th        12 non-null     str    
 3   cm_mean_c       12 non-null     float64
 4   bkk_mean_c      12 non-null     float64
 5   phkt_mean_c     12 non-null     float64
 6   cm_rain_mm      12 non-null     float64
 7   bkk_rain_mm     12 non-null     float64
 8   phkt_rain_mm    12 non-null     float64
 9   cm_rain_days    12 non-null     float64
 10  bkk_rain_days   12 non-null     float64
 11  phkt_rain_days  12 non-null     float64
dtypes: float64(9), int64(1), str(2)
memory usage: 1.3 KB


In [18]:
# describe() : สถิติสรุปทุกคอลัมน์ตัวเลขในคำสั่งเดียว
print(df.describe().round(2))

       month  cm_mean_c  bkk_mean_c  phkt_mean_c  cm_rain_mm  bkk_rain_mm  \
count  12.00      12.00       12.00        12.00       12.00        12.00   
mean    6.50      26.17       28.96        28.51       94.93       141.85   
std     3.61       2.41        1.05         0.65       79.51       114.07   
min     1.00      22.00       27.30        27.60        9.60        11.60   
25%     3.75      24.50       28.40        27.98       18.82        39.35   
50%     6.50      26.90       28.90        28.35       86.25       141.50   
75%     9.25      27.55       29.70        29.02      153.32       219.38   
max    12.00      29.50       30.70        29.60      223.30       335.90   

       phkt_rain_mm  cm_rain_days  bkk_rain_days  phkt_rain_days  
count         12.00         12.00          12.00           12.00  
mean         189.29          7.63           8.78           12.16  
std          114.15          6.15           6.37            5.95  
min           24.00          0.70     

## 🎯 ภารกิจบทที่ 1 (ส่งคำตอบใน Moodle)

เชียงใหม่มีอุณหภูมิเฉลี่ยรายวันในแต่ละเดือนต่างกันตามฤดู — อยากรู้ว่า **ทั้งปี 12 เดือน เชียงใหม่เย็น/ร้อนเฉลี่ยกี่องศา**

รันเซลล์ด้านล่างแล้วนำคำตอบ (ทศนิยม 2 ตำแหน่ง) ไปกรอกใน Moodle

In [19]:
# ภารกิจ 1: อุณหภูมิเฉลี่ยรายเดือนของเชียงใหม่ ทั้งปี (ทศนิยม 2 ตำแหน่ง)
df.info()
answer = round(df["cm_mean_c"].mean(), 2)
print("คำตอบภารกิจบทที่ 1 =", answer)

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   month           12 non-null     int64  
 1   month_en        12 non-null     str    
 2   month_th        12 non-null     str    
 3   cm_mean_c       12 non-null     float64
 4   bkk_mean_c      12 non-null     float64
 5   phkt_mean_c     12 non-null     float64
 6   cm_rain_mm      12 non-null     float64
 7   bkk_rain_mm     12 non-null     float64
 8   phkt_rain_mm    12 non-null     float64
 9   cm_rain_days    12 non-null     float64
 10  bkk_rain_days   12 non-null     float64
 11  phkt_rain_days  12 non-null     float64
dtypes: float64(9), int64(1), str(2)
memory usage: 1.3 KB
คำตอบภารกิจบทที่ 1 = 26.17
